In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

# 1. Định nghĩa cấu trúc thư mục (Dựa trên thực tế folder của bạn)
DATA_DIR = "/Users/huongnguyen/Library/CloudStorage/GoogleDrive-hainamnguyen0104@gmail.com/Drive của tôi/NGHIÊN CỨU KHOA HỌC 2526/Data"
FOLDERS = {
    "pdf": os.path.join(DATA_DIR, "PDF_Files"),
    "word": os.path.join(DATA_DIR, "Word_Files"),
    "excel": os.path.join(DATA_DIR, "Data_Files"),
    "powerpoint": os.path.join(DATA_DIR, "PowerPoint_Files"),
    "images": os.path.join(DATA_DIR, "Image_Files"),
    "quiz": os.path.join(DATA_DIR, "Quiz_Files")
}

# 2. Hàm quét và báo cáo số lượng file hiện có
def scan_data_folders():
    print("=== BÁO CÁO DỮ LIỆU HIỆN CÓ ===")
    summary = {}
    
    for key, path in FOLDERS.items():
        if os.path.exists(path):
            # Lấy danh sách các file (loại bỏ các file ẩn hệ thống như .DS_Store)
            files = [f for f in os.listdir(path) if not f.startswith('.')]
            summary[key] = files
            print(f"Thư mục {key.upper()}: Tìm thấy {len(files)} file.")
            for f in files[:3]: # Hiển thị tối đa 3 file mẫu cho mỗi loại
                print(f"   ∟  {f}")
            if len(files) > 3: print("   ∟ ...")
        else:
            print(f"Cảnh báo: Thư mục {path} không tồn tại!")
    
    return summary

# 3. Chạy lệnh quét
current_data = scan_data_folders()

=== BÁO CÁO DỮ LIỆU HIỆN CÓ ===
Thư mục PDF: Tìm thấy 2 file.
   ∟  TiengAnh1.pdf
   ∟  GiaoTrinhKinhTeChinhTri.pdf
Thư mục WORD: Tìm thấy 0 file.
Thư mục EXCEL: Tìm thấy 0 file.
Thư mục POWERPOINT: Tìm thấy 0 file.
Thư mục IMAGES: Tìm thấy 0 file.
Thư mục QUIZ: Tìm thấy 0 file.


In [ ]:
import os
import hashlib
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader, Docx2txtLoader, UnstructuredPowerPointLoader, 
    UnstructuredExcelLoader, CSVLoader
)

# 1. KHỞI TẠO CẤU HÌNH ĐƯỜNG DẪN
load_dotenv()
BASE_DIR = Path("/Users/huongnguyen/Desktop/NCKH")
DATA_DIR = Path("/Users/huongnguyen/Library/CloudStorage/GoogleDrive-hainamnguyen0104@gmail.com/Drive của tôi/NGHIÊN CỨU KHOA HỌC 2526/Data")

# Gom tất cả file cấu hình vào folder config
CONFIG_DIR = BASE_DIR / "config"

DB_DIR = CONFIG_DIR / "vector_db"

TRACKING_FILE = CONFIG_DIR / "data_registry.json"

def get_file_hash(file_path):
    """Tạo vân tay MD5 cho nội dung file"""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def incremental_embed():
    # 2. TẢI REGISTRY HIỆN CÓ
    if TRACKING_FILE.exists():
        with open(TRACKING_FILE, 'r', encoding='utf-8') as f:
            registry = json.load(f)
    else:
        registry = {}

    new_docs = []
    updated_registry = registry.copy()
    
    print("🔍 Đang quét sự thay đổi tài liệu...")
    
    # 3. QUÉT SUB-FOLDERS VÀ SO SÁNH HASH
    for root, dirs, files in os.walk(DATA_DIR):
        for file in files:
            if file.lower().endswith(('.pdf', '.docx', '.xlsx', '.pptx', '.csv')):
                full_path = Path(root) / file
                # CHỖ QUAN TRỌNG: Sử dụng đường dẫn tương đối để registry gọn gàng
                rel_path = str(full_path.relative_to(DATA_DIR))
                current_hash = get_file_hash(full_path)
                
                if rel_path not in registry or registry[rel_path] != current_hash:
                    print(f"🆕 Phát hiện mới/thay đổi: {rel_path}")
                    loader = None
                    if file.lower().endswith('.pdf'):
                        loader = PyPDFLoader(str(full_path))
                    elif file.lower().endswith(('.docx', '.doc')):
                        loader = Docx2txtLoader(str(full_path))
                    elif file.lower().endswith(('.pptx', '.ppt')):
                        loader = UnstructuredPowerPointLoader(str(full_path))
                    elif file.lower().endswith(('.xlsx', '.xls')):
                        loader = UnstructuredExcelLoader(str(full_path), mode="elements")
                    elif file.lower().endswith('.csv'):
                        loader = CSVLoader(str(full_path))

                    if loader:
                        try:
                            pages = loader.load()
                            for page in pages:
                                page.metadata["file_type"] = file.split('.')[-1]
                                page.metadata["source_file"] = rel_path
                            new_docs.extend(pages)
                            updated_registry[rel_path] = current_hash
                            print(f"✅ Đã nạp thành công: {file}")
                        except Exception as e:
                            print(f"❌ Lỗi xử lý {file}: {e}")

    # 4. TIẾN HÀNH EMBEDDING NẾU CÓ DỮ LIỆU MỚI
    if new_docs:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = text_splitter.split_documents(new_docs)
        
        batch_size = 10 
        print(f">>> Bắt đầu nhúng {len(chunks)} đoạn văn...")
        
        vector_db = Chroma(
            embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
            persist_directory=str(DB_DIR)
        )

        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            vector_db.add_documents(batch)
            print(f"... Đã xong đợt {i//batch_size + 1}")
            time.sleep(5) # Tránh Rate Limit
            
        # CẬP NHẬT REGISTRY SAU KHI EMBED THÀNH CÔNG
        with open(TRACKING_FILE, 'w', encoding='utf-8') as f:
            json.dump(updated_registry, f, indent=4, ensure_ascii=False)
        print("✨ Mọi thứ đã được đồng bộ hóa vào Database và Registry!")
    else:
        print("✅ Không có dữ liệu mới. Hệ thống đã đồng bộ.")

# 5. THỰC THI
incremental_embed()

🔍 Đang quét sự thay đổi tài liệu...
✅ Không có dữ liệu mới. Hệ thống đã đồng bộ.


In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# 1. Kết nối Vector DB
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_db = Chroma(
    persist_directory="config/vector_db", 
    embedding_function=embeddings
)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Khởi tạo LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.3)

# 3. Định nghĩa Prompt (Cá nhân hóa giáo dục)
system_prompt = (
    # [PERSONA]: Thiết lập vai trò
    "Bạn là một chuyên gia Học liệu số (Digital Learning Specialist), "
    "được thiết kế riêng để hỗ trợ sinh viên tạo tài liệu học tập như sinh bài giảng, câu hỏi ôn tập hoặc hình ảnh minh hoạ để học tập tại Đại học Mỏ - Địa Chất. "
    
    # [CONTEXT]: Bối cảnh dữ liệu
    "\n\nBỐI CẢNH: Bạn có quyền truy cập vào kho tri thức đa định dạng bao gồm giáo trình (PDF), "
    "bài tập thực hành (Word/Excel) và bài giảng trình chiếu (PPT) của nhóm"
    "Có thể truy cập vào bộ câu hỏi ôn tập (Quiz), và hình ảnh minh họa (Images) để hỗ trợ quá trình học tập. "
    "Dữ liệu tham khảo cụ thể như sau:\n{context}\n"
    
    # [TASK]: Nhiệm vụ cụ thể
    "\nNHIỆM VỤ: Dựa trên dữ liệu được cung cấp, hãy phân tích và trả lời câu hỏi của người học. "
    "Nếu thông tin không có trong tài liệu, hãy thành thật trả lời 'Tôi không tìm thấy thông tin này trong kho học liệu'."
    
    # [FORMAT]: Định dạng đầu ra mong muốn
    "\nĐỊNH DẠNG PHẢN HỒI:"
    "\n1. Câu trả lời cần súc tích, học thuật nhưng dễ hiểu."
    "\n2. Phải trích dẫn rõ tên file và trang (nếu có) từ tài liệu tham khảo."
    "\n3. Nếu có công thức toán học, hãy trình bày rõ ràng bằng định dạng văn bản dễ đọc."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 4. Tạo Chain hiện đại với Retrieval-Augmented Generation (RAG)
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("Hệ thống RAG hiện đại đã sẵn sàng!")

Hệ thống RAG hiện đại đã sẵn sàng!


/var/folders/r0/rl0ngg91681ggmyxzy2jqckr0000gn/T/ipykernel_87011/3502681532.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(
